In [9]:
pip install mlxtend --upgrade

Note: you may need to restart the kernel to use updated packages.


In [1]:
import ast
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

In [2]:
basket = pd.read_csv('customer_basket.csv')
clusters = pd.read_csv('cluster_assignments.csv')
basket = basket.merge(clusters, on='customer_id')

print(f"Total baskets: {len(basket)}")
print(f"Baskets per cluster:\n{basket['cluster'].value_counts().sort_index()}")

Total baskets: 100000
Baskets per cluster:
cluster
0     9486
1    14509
2    37885
3    31595
4     6525
Name: count, dtype: int64


In [3]:
basket['list_of_goods'] = basket['list_of_goods'].apply(ast.literal_eval)

# Quick sanity check
print(f"Sample basket: {basket['list_of_goods'].iloc[0]}")
print(f"Sample cluster: {basket['cluster'].iloc[0]}")

Sample basket: ['chicken', 'rice', 'pepper', 'whole wheat rice', 'shrimp', 'toothpaste', 'gums', 'cereals', 'bluetooth headphones', 'vacuum cleaner', 'body spray']
Sample cluster: 3


In [4]:
def get_association_rules(transactions, min_support=0.02, min_confidence=0.2):
    te = TransactionEncoder()
    te_array = te.fit_transform(transactions)
    df = pd.DataFrame(te_array, columns=te.columns_)
    frequent_itemsets = apriori(df, min_support=min_support, use_colnames=True)
    if frequent_itemsets.empty:
        print("No frequent itemsets found")
        return pd.DataFrame()
    rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.0)
    return rules.sort_values('lift', ascending=False)

In [5]:
all_products = [item for sublist in basket['list_of_goods'] for item in sublist]
print(f"Unique products: {len(set(all_products))}")
print(f"Most common products:\n{pd.Series(all_products).value_counts().head(10)}")

Unique products: 164
Most common products:
asparagus       12811
airpods         12145
cereals          9952
fresh bread      9934
butter           9654
eggs             9241
protein bar      8695
cooking oil      8623
toilet paper     8395
babies food      8318
Name: count, dtype: int64


In [6]:
print(basket['cluster'].value_counts().sort_index())
print(basket['cluster'].isna().sum())

cluster
0     9486
1    14509
2    37885
3    31595
4     6525
Name: count, dtype: int64
0


In [7]:
print(basket[basket['cluster'].isna()].shape)
basket = basket.dropna(subset=['cluster'])
basket['cluster'] = basket['cluster'].astype(int)
print(basket['cluster'].value_counts().sort_index())

(0, 4)
cluster
0     9486
1    14509
2    37885
3    31595
4     6525
Name: count, dtype: int64


In [8]:
# 4. Run per cluster and store results
cluster_rules = {}
for cluster_id in sorted(basket['cluster'].unique()):
    transactions = basket[basket['cluster'] == cluster_id]['list_of_goods'].tolist()
    rules = get_association_rules(transactions)
    cluster_rules[cluster_id] = rules

In [9]:
for cluster_id in sorted(basket['cluster'].unique()):
    print(f"\n{'='*50}")
    print(f"CLUSTER {cluster_id} — Top Unique Rules by Lift")
    print(f"{'='*50}")
    rules = cluster_rules[cluster_id]
    if not rules.empty:
        # Convert frozensets to sorted tuples to identify duplicates
        rules = rules.copy()
        rules['pair'] = rules.apply(
            lambda row: tuple(sorted([
                str(sorted(row['antecedents'])), 
                str(sorted(row['consequents']))
            ])), axis=1
        )
        # Keep only one direction per pair
        unique_rules = rules.drop_duplicates(subset='pair')
        top10 = unique_rules.nlargest(10, 'lift')[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
        display(top10)


CLUSTER 0 — Top Unique Rules by Lift


,antecedents,consequents,support,confidence,lift
576,"(green tea, eggs)",(cereals),0.021189,0.483173,1.367765
578,"(eggs, cereals)",(green tea),0.021189,0.146608,1.360789
580,(eggs),"(green tea, cereals)",0.021189,0.061884,1.349503
540,"(butter, oatmeal)",(milk),0.020346,0.260811,1.343863
606,"(eggs, salt)",(cereals),0.026460,0.472693,1.338098
778,(tea),"(fresh bread, honey)",0.022032,0.107455,1.330703
603,(eggs),"(cereals, oil)",0.021611,0.063116,1.330480
677,(honey),"(cereals, oatmeal)",0.022349,0.109959,1.328747
685,"(cereals, milk)",(oatmeal),0.021189,0.271989,1.328573
673,"(cereals, honey)",(oatmeal),0.022349,0.270754,1.322537



CLUSTER 1 — Top Unique Rules by Lift


,antecedents,consequents,support,confidence,lift
178,"(asparagus, carrots)",(salad),0.027638,0.438731,3.560149
161,(salad),"(asparagus, avocado)",0.026673,0.216443,3.516653
159,"(salad, asparagus)",(avocado),0.026673,0.407368,3.470645
153,"(asparagus, carrots)",(avocado),0.025501,0.404814,3.448882
197,(salad),"(asparagus, spinach)",0.027087,0.219799,3.410758
169,(avocado),"(asparagus, spinach)",0.025364,0.216089,3.353197
202,"(asparagus, tomatoes)",(salad),0.026397,0.409626,3.323970
175,(avocado),"(asparagus, tomatoes)",0.024950,0.212566,3.298525
186,(spinach),"(asparagus, carrots)",0.027431,0.206325,3.275232
79,(energy drink),(bluetooth headphones),0.025570,0.280211,3.268158



CLUSTER 2 — Top Unique Rules by Lift


,antecedents,consequents,support,confidence,lift
3,(energy drink),(airpods),0.023888,0.311961,2.612435
12,(cooking oil),(napkins),0.020298,0.229142,2.582105
1,(bluetooth headphones),(airpods),0.022331,0.297468,2.491067
14,(dog food),(napkins),0.022278,0.215581,2.429295
4,(babies food),(cooking oil),0.021724,0.214156,2.417546
9,(napkins),(babies food),0.021592,0.243308,2.398571
7,(dog food),(babies food),0.023967,0.231928,2.286394
11,(cooking oil),(dog food),0.020509,0.231526,2.240447



CLUSTER 3 — Top Unique Rules by Lift


,antecedents,consequents,support,confidence,lift
11,(deodorant),(shower gel),0.024877,0.283652,3.445593
17,(shower gel),(shampoo),0.024782,0.301038,3.441135
22,(tooth brush),(shower gel),0.024276,0.274026,3.328668
9,(deodorant),(shampoo),0.024846,0.283291,3.238273
18,(shampoo),(tooth brush),0.024687,0.282200,3.185459
25,(toothpaste),(shower gel),0.024687,0.252672,3.069276
12,(tooth brush),(deodorant),0.023675,0.267238,3.047057
26,(tooth brush),(toothpaste),0.025859,0.291890,2.987452
20,(shampoo),(toothpaste),0.025162,0.287627,2.943817
14,(deodorant),(toothpaste),0.024403,0.278239,2.847735



CLUSTER 4 — Top Unique Rules by Lift


,antecedents,consequents,support,confidence,lift
0,(asparagus),(airpods),0.021762,0.137464,1.058974


**Support** - proportion of baskets that contain both items

**Confidence** - percentage of bought together ('If someone buys item A, we have a x% confidence that buys item B)

**Lift** - measure of how much more likely someone is to buy two items together compared to independently
- Lift = 1 → no association, purely coincidental
- Lift > 1 → positive association, bought together more than expected
- Lift < 1 → negative association, rarely bought together

# Cluster Campaigns

**Cluster 0 - "Big Family Shopper"**
- eggs + green tea → cereals (lift 1.37), cereals + eggs → green tea (lift 1.36), honey → cereals + outmeal (lift 1.32) - Strong healthy breakfast pattern
- high drinks (alcohol and non-alcohol) and hygiene spending
- Campaign: **"Family Breakfast Bundle - buy cereals + green tea + eggs and get honey and outmeal 30% off"**, **"Party Ready Bundle - buy 3 alcoholic drinks and get 20% off non-alcoholic drinks"** and **"Family Hygiene Pack - buy €30 in hygiene products and get the cheapest one free"**

**Cluster 1 - "Bargain Hunter"**
- asparagus + carrots → salad (lift 3.56), salad → asparagus + avocado (lift 3.52), asparagus + carrots → avocado (lift 3.45), spinach → asparagus + carrots (lift 3.28), bluetooth headphones → energy drink (lift 3.27)
- spend mostly on sales
- Campaign: **"Healthy Plate Deal - buy asparagus + salad + spinach and get carrots + avocado at half price"** and **"Double Promotion Day — every Wednesday all your usual promoted items get an extra 10% off"**

**Cluster 2 - "Most Loyal"**
- energy drink → airpods (lift 2.61), bluetooth headphones → airpods (lift 2.49)
- high fresh_food_ratio, hygiene and tech spending 
- has_loyalty_card is 1
- Campaign: **"Tech Bundle - buy AirPods and get a free Energy Drink"** or **"Loyal Member Reward - use your loyalty card and get 10% off everything from the categories 'Fresh Food', 'Hygiene' and 'Tech'"**

maybe change category each week

**Cluster 3 - "New Customers + Tech Enthusiast"**
- Surprisingly dominated by hygiene products - shower gel → shampoo (lift 3.44), deodorant → shower gel (lift 3.44), tooth brush → shower gel (lift 3.33), shampoo → tooth brush (lift 3.19)
- high fresh_food_ratio and tech spending 
- Campaign: **"Welcome Hygiene Kit - buy shampoo + shower gel + deodorant and get a toothbrush free"** and **"Fresh Tech Deal - buy €50 in fresh food and get 10% off any electronics"**

maybe add incentive to create loyalty card

**Cluster 4 - "Biggest Buyers"**
- Only 1 meaningful rule (asparagus → airpods, lift 1.07) which is very weak
- Don't base the campaign on association rules here - use their spending profile instead
- Campaign: **"VIP Exclusive — spend over €150 and get 15% off your entire basket"**
